In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

In [2]:
def load_data(path):
    df = pd.read_csv(path)
    print("Data loaded successfully!")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    return df

In [3]:
def data_info(df):
    print(df.info())

In [4]:
def drop_weak_features(df, target_col="target", corr_threshold=0.1):
    corr = df.corr()[target_col].abs()
    weak = [col for col in corr.index if col != target_col and corr[col] < corr_threshold]
    return df.drop(columns=weak)

In [5]:
def split_data(df):
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

In [6]:
def split_train_test(X, y, test_size, random_state):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

In [7]:
def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    joblib.dump(scaler, "scaler.pkl")
    return X_train_scaled, X_test_scaled

In [8]:
def train_model(X_train, y_train):
    model = LogisticRegression(C=0.1, max_iter=5000)
    model.fit(X_train, y_train)
    return model


In [9]:
def test_model(model, X, y, label=""):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    print(f"Accuracy on {label} set: {acc:.4f}")
    return preds

In [10]:
def save_model(model, path):
    joblib.dump(model, path)
    print(f"Model saved at {path}")

In [11]:
def load_model(path):
    if os.path.exists(path):
        return joblib.load(path)
    else:
        raise FileNotFoundError(f"No model at {path}")

In [12]:
def predict_output(model, features):
    features = features.reshape(1, -1)
    op = model.predict(features)[0]
    return "Disease" if op == 1 else "No Disease"

In [13]:
Model_path = "HeartDisease_model.pkl"
Data_path = "/content/heart.csv"
TEST_SIZE = 0.2
RANDOM_STATE = 42
CORR_THRESHOLD = 0.1

df = load_data(Data_path)
df.info()
df = drop_weak_features(df, target_col="target", corr_threshold=CORR_THRESHOLD)

X, y = split_data(df)

X_train, X_test, y_train, y_test = split_train_test(X, y, TEST_SIZE, RANDOM_STATE)

X_train, X_test = scale_data(X_train, X_test)

model = train_model(X_train, y_train)

train_preds = test_model(model, X_train, y_train, label="Training")
test_preds = test_model(model, X_test, y_test, label="Testing")


save_model(model, Model_path)

Data loaded successfully!
Shape: (303, 14)
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB
Accuracy on Training set: 0.8554
Accuracy on

In [14]:
Model_path = "HeartDisease_model.pkl"
loaded_model = load_model(Model_path)

example_features = X_test[0]
pred_class = predict_output(loaded_model, example_features)

print("Predicted Class:", pred_class)
print("Actual Class:", y_test[0])


Predicted Class: No Disease
Actual Class: 0


In [15]:
!pip install streamlit -q
!pip install pyngrok
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 121.5 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [16]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Page configuration
st.set_page_config(page_title="Heart Disease Classifier", page_icon="❤️")

# Load the saved model and scaler
@st.cache_resource
def load_model_assets():
    model = joblib.load("HeartDisease_model.pkl")
    scaler = joblib.load("scaler.pkl")
    return model, scaler

def main():
    st.title("❤️ Heart Disease Prediction")
    st.markdown("--- ")

    try:
        model, scaler = load_model_assets()

        st.sidebar.header("Patient Input Features")

        # Create inputs for the features (adjust these based on the 11 features kept after dropping weak ones)
        # Note: In your notebook, you dropped weak features based on correlation.
        # The following inputs should represent the columns in your processed 'df'.

        age = st.sidebar.number_input("Age", 1, 100, 50)
        sex = st.sidebar.selectbox("Sex", options=[0, 1], format_func=lambda x: "Male" if x == 1 else "Female")
        cp = st.sidebar.slider("Chest Pain Type (cp)", 0, 3, 1)
        trestbps = st.sidebar.number_input("Resting Blood Pressure", 80, 200, 120)
        restecg = st.sidebar.slider("Resting Electrocardiographic Results", 0, 2, 0)
        thalach = st.sidebar.number_input("Maximum Heart Rate Achieved", 60, 220, 150)
        exang = st.sidebar.selectbox("Exercise Induced Angina", options=[0, 1])
        oldpeak = st.sidebar.number_input("ST Depression (Oldpeak)", 0.0, 6.0, 1.0)
        slope = st.sidebar.slider("Slope of Peak Exercise ST Segment", 0, 2, 1)
        ca = st.sidebar.slider("Number of Major Vessels", 0, 4, 0)
        thal = st.sidebar.slider("Thal", 0, 3, 2)

        # Organize input data into a dataframe or array
        # Ensure the order matches exactly with the training features
        input_data = np.array([[age, sex, cp, trestbps, restecg, thalach, exang, oldpeak, slope, ca, thal]])

        # Scale the data
        scaled_input = scaler.transform(input_data)

        if st.button("Predict"):
            # Model probability mapping: index 0 = Disease, index 1 = Healthy
            probabilities = model.predict_proba(scaled_input)[0]
            prob_disease = probabilities[0] * 100
            prob_healthy = probabilities[1] * 100

            prediction = model.predict(scaled_input)[0]

            # In this dataset: 0 = Heart Disease Present, 1 = Healthy
            if prediction == 0:
                st.error(f"### Result: High Risk of Heart Disease ({prob_disease:.1f}% estimated risk)")
            else:
                st.success(f"### Result: Low Risk / Healthy ({prob_healthy:.1f}% confidence)")

    except Exception as e:
        st.error(f"Error loading model or performing prediction: {e}")

if __name__ == '__main__':
    main()

Writing app.py
